# Comparing Ansible variable precedence

Ansible resolves variable names by checking multiple sources in a defined order. Lower-precedence sources get overridden by higher ones. This notebook tests the hierarchy step by step using `ansible-inventory` and `ansible-playbook` to see which value actually wins.

## Precedence order (lowest → highest)

1. Role defaults
2. Inventory vars (`group_vars`, `host_vars`)
3. Playbook vars
4. `include_vars` / `vars_files`
5. Task vars / `set_fact`
6. `extra vars` (`-e` flag)

> This isn't every source in the [official docs](https://docs.ansible.com/ansible/latest/playbook_guide/playbooks_variables.html#variable-precedence-where-should-i-put-a-variable), but it covers the ones a beginner hits first.

## Setup

Create a temporary directory with a minimal inventory, a single playbook, and a role.

In [ ]:
import tempfile, os, textwrap

tmp = tempfile.mkdtemp(prefix="ansible_precedence_")
print(f"Working in {tmp}")

# inventory with group_vars and host_vars
os.makedirs(f"{tmp}/inventory/group_vars/all")
os.makedirs(f"{tmp}/inventory/host_vars/localhost")

with open(f"{tmp}/inventory/hosts", "w") as f:
    f.write("[all]\nlocalhost ansible_connection=local\n")

with open(f"{tmp}/inventory/group_vars/all/msg.yml", "w") as f:
    f.write("msg: from group_vars/all\n")

with open(f"{tmp}/inventory/host_vars/localhost/msg.yml", "w") as f:
    f.write("msg: from host_vars/localhost\n")

# playbook
with open(f"{tmp}/playbook.yml", "w") as f:
    f.write(textwrap.dedent('''\
        - hosts: localhost
          gather_facts: no
          tasks:
            - debug:
                var: msg
    '''))

print("Created inventory, group_vars, host_vars, and playbook")
print(f"\nExpected: host_vars wins over group_vars")

## Test 1: group\_vars vs host\_vars

Both `group_vars/all/msg.yml` and `host_vars/localhost/msg.yml` define the same variable `msg`. According to the precedence docs, `host_vars` should win.

In [ ]:
import subprocess

result = subprocess.run(
    ["ansible-playbook", "-i", f"{tmp}/inventory", f"{tmp}/playbook.yml"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

The output should show `msg: from host_vars/localhost`. If ansible-playbook isn't available, the next best approach is to check `ansible-inventory`.

## Test 2: playbook vars override group\_vars

Add a `vars` block at the playbook level and see it beat `group_vars/all`.

In [ ]:
# rewrite playbook with playbook-level vars
with open(f"{tmp}/playbook.yml", "w") as f:
    f.write(textwrap.dedent('''\
        - hosts: localhost
          gather_facts: no
          vars:
            msg: from playbook vars
          tasks:
            - debug:
                var: msg
    '''))

result = subprocess.run(
    ["ansible-playbook", "-i", f"{tmp}/inventory", f"{tmp}/playbook.yml"],
    capture_output=True, text=True
)
print("Expect 'from playbook vars' — playbook vars > group_vars")
print(result.stdout)

## Test 3: role defaults vs role vars

Role defaults (`defaults/main.yml`) have the lowest precedence. Role-level vars (`vars/main.yml`) sit much higher. This test creates a role with both and sees which one survives.

In [ ]:
# create a role with both defaults and vars
role_dir = f"{tmp}/roles/test_role"
os.makedirs(f"{role_dir}/defaults")
os.makedirs(f"{role_dir}/vars")
os.makedirs(f"{role_dir}/tasks")

with open(f"{role_dir}/defaults/main.yml", "w") as f:
    f.write("role_msg: from role defaults\n")

with open(f"{role_dir}/vars/main.yml", "w") as f:
    f.write("role_msg: from role vars/main.yml\n")

with open(f"{role_dir}/tasks/main.yml", "w") as f:
    f.write(textwrap.dedent('''\
        - debug:
            var: role_msg
    '''))

# playbook that uses the role
with open(f"{tmp}/playbook_role.yml", "w") as f:
    f.write(textwrap.dedent('''\
        - hosts: localhost
          gather_facts: no
          roles:
            - test_role
    '''))

result = subprocess.run(
    ["ansible-playbook", "-i", f"{tmp}/inventory", f"{tmp}/playbook_role.yml"],
    capture_output=True, text=True
)
print("Expect 'from role vars/main.yml' — role vars > role defaults")
print(result.stdout)

## Test 4: extra vars override everything

Extra vars (`-e` on the command line) sit at the top of the precedence stack. Even playbook vars or `set_fact` can't beat them in the same play.

In [ ]:
result = subprocess.run(
    [
        "ansible-playbook", "-i", f"{tmp}/inventory", f"{tmp}/playbook.yml",
        "-e", "msg=from extra vars"
    ],
    capture_output=True, text=True
)
print("Expect 'from extra vars' — extra vars beat playbook vars")
print(result.stdout)

## Summary

| Precedence level | Source | Wins against |
|---|---|---|
| Lowest | Role defaults | — |
| Low | `group_vars/all` | Role defaults |
| Medium | `host_vars/<host>` | group_vars |
| Medium-high | Playbook vars | host_vars |
| High | Role vars (`vars/main.yml`) | Role defaults |
| Highest | Extra vars (`-e`) | Everything |

The trick to remembering this: `extra > playbook > inventory > role defaults`. The docs list ~22 sources, but the ones above cover 80\% of the confusion beginners hit.

> One thing I'm still not sure about: where `set_fact` lands relative to `include_vars` in the middle of the stack. The docs say `set_fact` beats `include_vars` when both run at task time, but I haven't tested the caching behavior across plays yet.